In [1]:
import re
import warnings
import requests
import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
def extract_variants(vcf_path: str) -> list[str]:
    """Extract unique protein-level variant names from a VEP-annotated VCF."""
    pattern = re.compile(r'p\.([A-Z][a-z]{2}\d+[A-Z][a-z]{2})')
    seen, variants = set(), []
    with open(vcf_path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            match = pattern.search(line)
            if match:
                name = match.group(1)
                if name not in seen:
                    seen.add(name)
                    variants.append(name)
    print(f"Found {len(variants)} unique variants")
    return variants

variants = extract_variants("All_Variants_VEP.Gene.vcf")

Found 3220 unique variants


In [3]:
# Download the latest CFTR2 variant classification spreadsheet
CFTR2_URL = "https://cftr2.org/sites/default/files/CFTR2_30January2026.xlsx"

resp = requests.get(CFTR2_URL, verify=False)
with open("cftr2_variants.xlsx", "wb") as f:
    f.write(resp.content)
print(f"Downloaded {len(resp.content):,} bytes")

Downloaded 353,148 bytes


In [4]:
# Data starts at row 12 (0-indexed header=11) — rows above are metadata
cftr2 = pd.read_excel("cftr2_variants.xlsx", header=11)
cftr2.columns = [
    "legacy_name", "protein_name", "cdna_name", "alt_names",
    "alleles_count", "allele_frequency", "determination_2024",
    "determination_2026", "change"
]
cftr2 = cftr2.dropna(subset=["protein_name"])
print(f"CFTR2: {len(cftr2)} variants loaded")

CFTR2: 2092 variants loaded


In [5]:
# VEP outputs three-letter amino acid codes without prefix (e.g. Ser13Phe)
# CFTR2 uses HGVS protein format with p. prefix (e.g. p.Ser13Phe)
variants_hgvs = [f"p.{v}" for v in variants]

vcf_df = pd.DataFrame({"variant": variants, "protein_name": variants_hgvs})
result = vcf_df.merge(
    cftr2[["protein_name", "legacy_name", "determination_2026", "allele_frequency"]],
    on="protein_name",
    how="left"
)

result.to_csv("cftr2_results.csv", index=False)

print(f"Total variants : {len(result)}")
print(f"Matched        : {result['determination_2026'].notna().sum()}")
print(f"Not in CFTR2   : {result['determination_2026'].isna().sum()}")
print()
print(result["determination_2026"].value_counts())

Total variants : 3220
Matched        : 656
Not in CFTR2   : 2564

determination_2026
No interpretation available     325
CF-causing                      226
Varying clinical consequence     72
Non CF-causing                   33
Name: count, dtype: int64
